# Treinamento: ResNet + BiLSTM

Prediz um *arousal score* contínuo por clipe a partir do vídeo em grayscale.
Cada experimento gera um diretório próprio em `runs/` para o TensorBoard.

**Decisões de treino:**
- `epochs=100` (teto pq tem *early stopping*).
- Checkpoint/early-stopping por **CCC**; scheduler observa a **val loss**.
- `batch_size=4` (`2` na ResNet50 por memória).
- *Gradient clipping* `1.0` e semente fixa.

## 1. Setup

In [ ]:
import os
import sys
import datetime
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from tqdm.auto import tqdm
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.tensorboard import SummaryWriter

from transformers import AutoModel # transformers==4.44.0
import einops
import timm
import torchaudio # torchaudio==2.5.1

sys.path.insert(0, "/mnt/storage_C4/gaussian_football")
sys.path.insert(0, "/mnt/storage_C4/gaussian_football/preprocessing")
sys.path.insert(0, "/mnt/storage_C4/gaussian_football/models/nn")

from resnetlstm import ResNetLSTM
from resnetlstm_multimodal import ResNetLSTM_MultiModal
#from loaders import build_video_dataloader, build_mel_dataloader
from multimodal_loader_and_dataset import build_multimodal_dataloader
#from balanced_dataset import balanced_df

ModuleNotFoundError: No module named 'torch'

In [ ]:
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))

SEED = 435
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Torch: 2.5.1+cu121
Torchvision: 0.20.1+cu121
CUDA disponível: True
CUDA: 12.1
GPU: NVIDIA RTX A4500
Device: cuda


## 2. Configuração

Metadados a serem ajustados.

In [ ]:
# paths
LABELS_ALL   = "/mnt/storage_C4/gaussian_football/data/labels/labels_all.csv"
LABELS_DIR   = "/mnt/storage_C4/gaussian_football/data/labels"
CKPT_DIR     = "/mnt/storage_C4/gaussian_football/models/checkpoints"
RUNS_DIR     = "/mnt/storage_C4/gaussian_football/runs_multimodal"  # logs do TensorBoard

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

# treino 
EPOCHS        = 100        # teto por causa do early stopping decide
PATIENCE      = 10         # épocas sem melhora antes de parar
LR            = 1e-4
WEIGHT_DECAY  = 1e-4
GRAD_CLIP     = 1.0
MONITOR       = "ccc"      # métrica de checkpoint/early-stopping: "ccc" | "mae" | "loss"

# dataloaders 
BATCH_SIZE = 2

# modelo padrão
MODEL_DEFAULTS = dict(
    frame_step=2,
    hidden_size=256,
    num_layers=1,
    use_dropout=False,
    dropout_p=0.3,
)

## 3. Balanceando os Dados

Gera os CSVs balanceados (50% highlight / 50% não-highlight por partida,
com `threshold=0.5` no `arousal_score`). Só recalcula se os arquivos não existirem
(use `FORCE_REBUILD = True` para refazer).

In [ ]:
FORCE_REBUILD = False

paths_balanced = {
    "train": os.path.join(LABELS_DIR, "balanced_labels_train.csv"),
    "valid": os.path.join(LABELS_DIR, "balanced_labels_val.csv"),
    "test":  os.path.join(LABELS_DIR, "balanced_labels_test.csv"),
}

if FORCE_REBUILD or not all(os.path.exists(p) for p in paths_balanced.values()):
    all_data = pd.read_csv(LABELS_ALL)
    for split, out_path in paths_balanced.items():
        subset = all_data[all_data["split"] == split]
        balanced = balanced_df(subset, "game_id", threshold=0.5, random_state=SEED)
        balanced.to_csv(out_path, index=False)
        print(f"{split}: {len(balanced)} clipes -> {out_path}")
else:
    print("CSVs balanceados já existem. (FORCE_REBUILD=True para refazer.)")

CSVs balanceados já existem. (FORCE_REBUILD=True para refazer.)


## 4. DataLoaders Multimodais

In [ ]:
TRAIN_PATH = f"{LABELS_DIR}/balanced_labels_train.csv"
VAL_PATH   = f"{LABELS_DIR}/balanced_labels_val.csv"

train_multimodal_loader = build_multimodal_dataloader(
    csv_path=TRAIN_PATH,
    split="train",
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    is_grayscale=True,
    pin_memory=True,    
)

valid_multimodal_loader = build_multimodal_dataloader(
    csv_path=VAL_PATH,
    split='valid',
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    is_grayscale=True,
    pin_memory=True,
)

NameError: name 'LABELS_DIR' is not defined

## 5. Métricas

O **CCC** (Concordance Correlation Coefficient) mede concordância entre predição e alvo,
punindo tanto erro de correlação quanto deslocamento de escala/média, por isso é sensível
ao *variance collapse* (modelo prevendo sempre perto da média).

$$\rho_c = \dfrac{2\,\rho\,\sigma_x\,\sigma_y}{\sigma_x^2 + \sigma_y^2 + (\mu_x - \mu_y)^2}$$

In [6]:
def calculate_ccc(y_true, y_pred):
    '''Concordance Correlation Coefficient sobre dois arrays 1D.'''
    mean_true, mean_pred = np.mean(y_true), np.mean(y_pred)
    var_true,  var_pred  = np.var(y_true),  np.var(y_pred)
    covariance = np.mean((y_true - mean_true) * (y_pred - mean_pred))
    denominator = var_true + var_pred + (mean_true - mean_pred) ** 2
    return 0.0 if denominator == 0 else (2 * covariance) / denominator

## 6. Funções de perda

- `mse`: minimiza o erro quadrático médio.
- `bce`: `BCEWithLogitsLoss`, trata o arousal score como probabilidade em `[0, 1]`, já tem sigmoide embutida.
- `ccc`: `1 - CCC`, força o modelo a preservar variância e correlação.
- `combined`: `MSE + alpha * (1 - CCC)`, combinando as coisas.

In [7]:
def ccc_loss(pred, target):
    pred_mean, target_mean = pred.mean(), target.mean()
    covariance = ((pred - pred_mean) * (target - target_mean)).mean()
    pred_var, target_var = pred.var(), target.var()
    ccc = (2 * covariance) / (pred_var + target_var + (pred_mean - target_mean) ** 2 + 1e-8)
    return 1 - ccc

def combined_loss(pred, target, alpha=0.5):
    return nn.MSELoss()(pred, target) + alpha * ccc_loss(pred, target)

LOSSES = {
    "mse":      nn.MSELoss(),
    "bce":      nn.BCEWithLogitsLoss(),
    "ccc":      ccc_loss,
    "combined": combined_loss,
}

## 7. Treino

Função de treino unificada.

In [ ]:
MONITOR_MODE = {"ccc": "max", "mae": "min", "loss": "min"}

def _apply_freeze(model, freeze_backbone):
    '''Liga/desliga o gradiente da ResNet. conv1 (índice 0) fica sempre treinável.'''
    for name, param in model.cnn.named_parameters():
        param.requires_grad = name.startswith("0.") or (not freeze_backbone)

def _set_backbone_eval(model):
    '''Congela estatísticas de BatchNorm na ResNet. Conv1 fica em train.'''
    for i, module in enumerate(model.cnn):
        module.train() if i == 0 else module.eval()

def _is_better(current, best, mode):
    return current > best if mode == "max" else current < best

def binary_accuracy(y_true, y_pred, threshold=0.5):
    return float(np.mean((y_pred > threshold) == (y_true > threshold)))

def _log_pred_scatter(writer, y_true, y_pred, tag, step):
    '''Scatter pred x target — faixa horizontal achatada = variance collapse.'''
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.scatter(y_true, y_pred, s=8, alpha=0.4)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lims, lims, "--", color="gray", linewidth=1)  # diagonal ideal
    ax.set_xlabel("target"); ax.set_ylabel("pred"); ax.set_title("pred vs target")
    writer.add_figure(tag, fig, step)
    plt.close(fig)


def train_model(
    model, train_loader, val_loader, *,
    criterion, run_name, optimizer, scheduler,
    backbone_name, loss_name,
    freeze_mode="finetune", unfreeze_epoch=5,
    epochs=EPOCHS, patience=PATIENCE, monitor=MONITOR,
    grad_clip=GRAD_CLIP, use_amp=False,
    checkpoint_path=None, device=device,
):
    '''Treina e valida por época, logando tudo no TensorBoard.

    Args:
        criterion: função de perda (vindo de LOSSES).
        run_name: nome do experimento (usado no log_dir e no checkpoint).
        backbone_name, loss_name: usados nos hparams do TensorBoard.
        freeze_mode: "full" | "frozen" | "finetune".
        unfreeze_epoch: época de descongelamento (só vale para "finetune").
        monitor: métrica de checkpoint/early-stopping ("ccc" | "mae" | "loss").
        use_amp: ativa mixed precision (mais rápido; cheque o ccc_loss em fp16).
    Returns:
        dict com as melhores métricas e o caminho do checkpoint.
    '''
    model.to(device)
    torch.cuda.empty_cache()
    if checkpoint_path is None:
        checkpoint_path = os.path.join(CKPT_DIR, f"{run_name}.pth")
    last_path = os.path.join(CKPT_DIR, f"{run_name}_last.pth")

    # com BCE a saída é logit: aplica sigmoid antes das métricas (não na loss)
    use_sigmoid = isinstance(criterion, nn.BCEWithLogitsLoss)

    mode = MONITOR_MODE[monitor]
    best_score = -np.inf if mode == "max" else np.inf
    best_metrics, best_true, best_pred = {}, None, None
    epochs_no_improve = 0

    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    log_dir = os.path.join(RUNS_DIR, f"{run_name}_{stamp}")
    writer = SummaryWriter(log_dir=log_dir)
    print(f"TensorBoard: {log_dir}")

    for epoch in range(epochs):
        # estado de congelamento desta época
        if freeze_mode == "full":
            backbone_frozen = False
        elif freeze_mode == "frozen":
            backbone_frozen = True
        else:  # finetune
            backbone_frozen = epoch < unfreeze_epoch
            if epoch == unfreeze_epoch:
                print(f"[época {epoch+1}] descongelando a ResNet (fine-tuning completo)")

        _apply_freeze(model, backbone_frozen)

        # ----- treino -----
        model.train()
        if backbone_frozen:
            _set_backbone_eval(model)

        train_loss = 0.0
        train_true, train_pred = [], []
        for videos, _, mel_spects, targets in tqdm(train_loader, desc=f"época {epoch+1}/{epochs} [treino]", leave=False):
            videos = videos.to(device, non_blocking=True)
            mel_spects = mel_spects.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True).float().view(-1)

            optimizer.zero_grad()
            with torch.amp.autocast("cuda", enabled=use_amp):
                outputs = model(videos, mel_spects).reshape(-1)
                loss = criterion(outputs, targets)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)  # unscale antes do clip
            torch.nn.utils.clip_grad_norm_(filter(lambda p: p.requires_grad, model.parameters()), max_norm=grad_clip)
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * videos.size(0)
            preds = torch.sigmoid(outputs) if use_sigmoid else outputs
            train_true.extend(targets.detach().cpu().numpy())
            train_pred.extend(preds.detach().cpu().numpy())
        train_loss /= len(train_loader.dataset)

        train_true, train_pred = np.array(train_true), np.array(train_pred)
        train_mae = np.mean(np.abs(train_true - train_pred))
        train_ccc = calculate_ccc(train_true, train_pred)

        # ----- validação -----
        model.eval()
        val_loss = 0.0
        all_true, all_pred = [], []
        with torch.no_grad():
            for videos, _, mel_spects, targets in tqdm(val_loader, desc=f"época {epoch+1}/{epochs} [val]", leave=False):
                videos = videos.to(device, non_blocking=True)
                mel_spects = mel_spects.to(device, non_blocking=True)
                targets = targets.to(device, non_blocking=True).float().view(-1)
                with torch.amp.autocast("cuda", enabled=use_amp):
                    outputs = model(videos, mel_spects).reshape(-1)
                    val_loss += criterion(outputs, targets).item() * videos.size(0)
                preds = torch.sigmoid(outputs) if use_sigmoid else outputs
                all_true.extend(targets.cpu().numpy())
                all_pred.extend(preds.cpu().numpy())
        val_loss /= len(val_loader.dataset)

        all_true, all_pred = np.array(all_true), np.array(all_pred)
        val_mae = np.mean(np.abs(all_true - all_pred))
        val_ccc = calculate_ccc(all_true, all_pred)
        pred_std = float(np.std(all_pred)) 
        target_std = float(np.std(all_true))

        scheduler.step(val_loss)

        # ----- tensorboard -----
        writer.add_scalar("Loss/train", train_loss, epoch)
        writer.add_scalar("Loss/val", val_loss, epoch)
        writer.add_scalar("Metrics/MAE_train", train_mae, epoch)
        writer.add_scalar("Metrics/MAE_val", val_mae, epoch)
        writer.add_scalar("Metrics/CCC_train", train_ccc, epoch)
        writer.add_scalar("Metrics/CCC_val", val_ccc, epoch)
        writer.add_scalar("Diagnostics/pred_std", pred_std, epoch)
        writer.add_scalar("Diagnostics/target_std", target_std, epoch)

        writer.add_histogram("Val/predictions", all_pred, epoch)
        for gi, pg in enumerate(optimizer.param_groups):
            writer.add_scalar(f"Train/LR_group{gi}", pg["lr"], epoch)

        print(
            f"época [{epoch+1}/{epochs}]"
            f" | loss  train {train_loss:.4f}  val {val_loss:.4f}"
            f" | MAE   train {train_mae:.4f}  val {val_mae:.4f}"
            f" | CCC   train {train_ccc:.4f}  val {val_ccc:.4f}"
            f" | pred_std {pred_std:.4f} | LR {optimizer.param_groups[0]['lr']:.2e}"
        )

        # ----- checkpoint (best + last) + early stopping -----
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "run_name": run_name}, last_path)

        current = {"ccc": val_ccc, "mae": val_mae, "loss": val_loss}[monitor]
        if _is_better(current, best_score, mode):
            best_score = current
            best_metrics = {"val_loss": val_loss, "val_mae": val_mae,
                            "val_ccc": val_ccc, "epoch": epoch}
            best_true, best_pred = all_true, all_pred
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_metrics": best_metrics,
                "run_name": run_name,
            }, checkpoint_path)
            print(f"  novo melhor ({monitor}={best_score:.4f}) salvo em {checkpoint_path}")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"early stopping (sem melhora por {patience} épocas)")
                break

    # scatter do melhor epoch — diagnóstico de variance collapse
    if best_pred is not None:
        _log_pred_scatter(writer, best_true, best_pred, "Val/pred_vs_target", best_metrics["epoch"])

    # hparams para comparar runs na aba HPARAMS (run_name="." liga aos scalars)
    writer.add_hparams(
        {"backbone": backbone_name, "loss": loss_name, "freeze_mode": freeze_mode,
         "lr": LR, "hidden_size": MODEL_DEFAULTS["hidden_size"],
         "num_layers": MODEL_DEFAULTS["num_layers"], "frame_step": MODEL_DEFAULTS["frame_step"]},
        {"best/val_ccc": best_metrics.get("val_ccc", 0.0),
         "best/val_mae": best_metrics.get("val_mae", 0.0),
         "best/val_loss": best_metrics.get("val_loss", 0.0)},
        run_name=".",
    )
    writer.close()
    print(f"concluído. melhor {monitor}: {best_score:.4f}")
    return {"checkpoint": checkpoint_path, "best_metrics": best_metrics}

## 8. `run_experiment`

Função reaproveitável.

In [9]:
def run_experiment(
    audiomae,
    backbone_name="resnet18",
    loss_name="bce",
    freeze_mode="finetune",
    unfreeze_epoch=5,
    lr=LR,
    lr_backbone=None,
    weight_decay=WEIGHT_DECAY,
    model_kwargs=None,
    epochs=EPOCHS,
    patience=PATIENCE,
    monitor=MONITOR,
    use_amp=False,
):
    '''Roda um experimento completo e retorna o resultado.'''
    model_kwargs = {**MODEL_DEFAULTS, **(model_kwargs or {})}
    run_name = f"{backbone_name}__{loss_name}__{freeze_mode}"
    print(f"\n=== {run_name} ===")

    model = ResNetLSTM_MultiModal(audiomae, backbone_name=backbone_name, **model_kwargs).to(device)

    if lr_backbone is None:
        optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = AdamW([
            {"params": model.cnn.parameters(),  "lr": lr_backbone},
            {"params": model.lstm.parameters(), "lr": lr},
            {"params": model.head.parameters(), "lr": lr},
        ], weight_decay=weight_decay)

    scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5)

    return train_model(
        model, train_multimodal_loader, valid_multimodal_loader,
        criterion=LOSSES[loss_name], run_name=run_name,
        optimizer=optimizer, scheduler=scheduler,
        backbone_name=backbone_name, loss_name=loss_name,
        freeze_mode=freeze_mode, unfreeze_epoch=unfreeze_epoch,
        epochs=epochs, patience=patience, monitor=monitor,
        use_amp=use_amp,
    )

## 9. Rodar experimentos

Altos experimentos.

In [10]:
# definindo o modelo para embedding dos mel spectrogramas
model_ae = AutoModel.from_pretrained("hance-ai/audiomae", trust_remote_code=True).to(device)

In [ ]:
results = {}

experimentos = (
    [("resnet18", loss, "frozen", 5) for loss in ["mse", "bce", "ccc"]]
    + [(backbone, loss, "frozen", 5) for backbone in ["resnet34", "resnet50"] for loss in ["mse", "bce", "ccc"]]
    + [("resnet18", loss, "finetune", 5) for loss in ["mse", "bce", "ccc"]]
)

for backbone, loss_name, freeze_mode, unfreeze_epoch in experimentos:
    key = f"{backbone}__{loss_name}__{freeze_mode}"
    print(f"\n>>> iniciando {key}")
    try:
        results[key] = run_experiment(
            audiomae=model_ae,
            backbone_name=backbone,
            loss_name=loss_name,
            freeze_mode=freeze_mode,
            unfreeze_epoch=unfreeze_epoch,
            epochs=100,
        )
    except Exception as e:
        print(f"ERRO em {key}: {e}")
        results[key] = None

print("\n=== RESUMO ===")
for key, result in results.items():
    if result is None:
        print(f"{key}: FALHOU")
    else:
        m = result["best_metrics"]
        print(f"{key}: CCC={m['val_ccc']:.4f} | MAE={m['val_mae']:.4f} | loss={m['val_loss']:.4f}")


>>> iniciando resnet18__mse__frozen

=== resnet18__mse__frozen ===
TensorBoard: /mnt/storage_C4/gaussian_football/runs_multimodal/resnet18__mse__frozen_20260616-172623


época 1/100 [treino]:   0%|          | 0/1471 [00:00<?, ?it/s]

ERRO em resnet18__mse__frozen: Sizes of tensors must match except in dimension 2. Expected size 45 but got size 1 for tensor number 1 in the list.

>>> iniciando resnet18__bce__frozen

=== resnet18__bce__frozen ===
TensorBoard: /mnt/storage_C4/gaussian_football/runs_multimodal/resnet18__bce__frozen_20260616-172624


época 1/100 [treino]:   0%|          | 0/1471 [00:00<?, ?it/s]

ERRO em resnet18__bce__frozen: Sizes of tensors must match except in dimension 2. Expected size 47 but got size 1 for tensor number 1 in the list.

>>> iniciando resnet18__ccc__frozen

=== resnet18__ccc__frozen ===


...

época 1/100 [treino]:   0%|          | 0/1471 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7728b9763ba0>
Traceback (most recent call last):
  File "/home/al.leticia.ferreira/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/home/al.leticia.ferreira/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1587, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/opt/anaconda3/envs/ambiente_aluno/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7728b9763ba0>
Traceback (most recent call last):
  File "/home/al.leticia.ferreira/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    Exception ignored

ERRO em resnet50__mse__frozen: Sizes of tensors must match except in dimension 2. Expected size 47 but got size 1 for tensor number 1 in the list.

>>> iniciando resnet50__bce__frozen

=== resnet50__bce__frozen ===
TensorBoard: /mnt/storage_C4/gaussian_football/runs_multimodal/resnet50__bce__frozen_20260616-172650


época 1/100 [treino]:   0%|          | 0/1471 [00:00<?, ?it/s]

ERRO em resnet50__bce__frozen: Sizes of tensors must match except in dimension 2. Expected size 45 but got size 1 for tensor number 1 in the list.

>>> iniciando resnet50__ccc__frozen

=== resnet50__ccc__frozen ===


In [ ]:
results = {}

experimentos = (
    [(backbone, loss, "frozen", 5) for backbone in ["resnet50"] for loss in ["mse", "bce", "ccc"]]
    + [("resnet18", loss, "finetune", 5) for loss in ["mse", "bce", "ccc"]]
)

for backbone, loss_name, freeze_mode, unfreeze_epoch in experimentos:
    key = f"{backbone}__{loss_name}__{freeze_mode}"
    print(f"\n>>> iniciando {key}")
    try:
        results[key] = run_experiment(
            audiomae=model_ae,
            backbone_name=backbone,
            loss_name=loss_name,
            freeze_mode=freeze_mode,
            unfreeze_epoch=unfreeze_epoch,
            epochs=100,
        )
    except Exception as e:
        print(f"ERRO em {key}: {e}")
        results[key] = None

print("\n=== RESUMO ===")
for key, result in results.items():
    if result is None:
        print(f"{key}: FALHOU")
    else:
        m = result["best_metrics"]
        print(f"{key}: CCC={m['val_ccc']:.4f} | MAE={m['val_mae']:.4f} | loss={m['val_loss']:.4f}")

In [ ]:
results = {}

experimentos = (
    [(backbone, loss, "finetune", 5) for backbone in ["resnet34","resnet50"]  for loss in ["mse", "bce", "ccc"]]
)

for backbone, loss_name, freeze_mode, unfreeze_epoch in experimentos:
    key = f"{backbone}__{loss_name}__{freeze_mode}"
    print(f"\n>>> iniciando {key}")
    try:
        results[key] = run_experiment(
            audiomae=model_ae,
            backbone_name=backbone,
            loss_name=loss_name,
            freeze_mode=freeze_mode,
            unfreeze_epoch=unfreeze_epoch,
            epochs=100,
        )
    except Exception as e:
        print(f"ERRO em {key}: {e}")
        results[key] = None

print("\n=== RESUMO ===")
for key, result in results.items():
    if result is None:
        print(f"{key}: FALHOU")
    else:
        m = result["best_metrics"]
        print(f"{key}: CCC={m['val_ccc']:.4f} | MAE={m['val_mae']:.4f} | loss={m['val_loss']:.4f}")

## 10. TensorBoard

Ver os resultados.

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir /mnt/storage_C4/gaussian_football/runs/